# Result · Patched minus undefended agreement on each test set

**In the paper ([arXiv:2610.10899](https://arxiv.org/abs/2610.10899)):** the Sec. V paragraph "Fidelity" (out-of-domain loss per test set, worst on TMHINT-QI).

System-level SRCC of R3 minus R0 on each of the 13 test sets, paired over seeds with Holm-corrected p-values (the
preregistered secondary analysis of E2). Recomputes the E2 evaluation of R0 and R3 (6 jobs, ~3 min each).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e2'] and v.get("predictor") in ("R0", "R3")]
print(len(KEYS), "runs:", ", ".join(KEYS))

6 runs: e2_R0_s0, e2_R0_s1, e2_R0_s2, e2_R3_s0, e2_R3_s1, e2_R3_s2


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    6
Name: count, dtype: int64

In [6]:
display(Markdown((OUT / "e2_r3_minus_r0.md").read_text()))

| corpus              |   n_seeds |   mean_diff | all_seeds_same_sign   |   p_raw |   p_holm |
|:--------------------|----------:|------------:|:----------------------|--------:|---------:|
| tmhintqi/test       |         3 |     -0.2960 | True                  |  0.0946 |   0.9463 |
| vmc23/track3        |         3 |     -0.2184 | True                  |  0.3004 |   1.0000 |
| nisqa/test_for      |         3 |     -0.1839 | True                  |  0.0498 |   0.5473 |
| nisqa/test_livetalk |         3 |     -0.1505 | True                  |  0.0264 |   0.3164 |
| vmc23/track2        |         3 |     -0.0715 | True                  |  0.0028 |   0.0363 |
| urgent25_sqa/test   |         3 |     -0.0636 | False                 |  0.2490 |   1.0000 |
| vmc23/track1a       |         3 |     -0.0325 | False                 |  0.4772 |   1.0000 |
| vmc23/track1b       |         3 |     -0.0286 | False                 |  0.6734 |   1.0000 |
| urgent24_sqa/test   |         3 |     -0.0250 | True                  |  0.1577 |   1.0000 |
| bvcc_ood/test       |         3 |     -0.0210 | True                  |  0.2809 |   1.0000 |
| somos/test          |         3 |     -0.0144 | False                 |  0.7180 |   0.7180 |
| nisqa/test_p501     |         3 |     -0.0142 | False                 |  0.2360 |   1.0000 |
| bvcc/test           |         3 |     -0.0018 | False                 |  0.5180 |   1.0000 |

In [7]:
repro.compare_csv(OUT / "e2_r3_minus_r0.csv", COMMITTED_OUT / "e2_r3_minus_r0.csv")

n_seeds                0.0
mean_diff              0.0
all_seeds_same_sign    0.0
p_raw                  0.0
p_holm                 0.0
Name: max_abs_diff, dtype: float64

In [8]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('ood_drop',))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

13 numbers; max |diff| = 0.0000; identical: 13


,paper,reproduced,abs_diff
ood_drop_bvcc/test,-0.002,-0.002,0.0
ood_drop_bvcc_ood/test,-0.021,-0.021,0.0
ood_drop_nisqa/test_for,-0.184,-0.184,0.0
ood_drop_nisqa/test_livetalk,-0.151,-0.151,0.0
ood_drop_nisqa/test_p501,-0.014,-0.014,0.0
ood_drop_somos/test,-0.014,-0.014,0.0
ood_drop_tmhintqi/test,-0.296,-0.296,0.0
ood_drop_urgent24_sqa/test,-0.025,-0.025,0.0
ood_drop_urgent25_sqa/test,-0.064,-0.064,0.0
ood_drop_vmc23/track1a,-0.032,-0.032,0.0
